In [1]:
# 分析対象銘柄の証券コードをセット
from datetime import date
code = 9997
ev_date = date.today()

In [2]:
# 読み込みファイルパスの設定とimportしたいmoduleパス(pythonパス)の設定
from pathlib import Path
import os

CURRENT_DIR = Path(os.getcwd())
PJ_DIR = CURRENT_DIR.parent.parent
DATA_DIR = PJ_DIR / "data" 


# notebook内で利用するmoduleのimport
from wequant.data_processing import KessanPl, read_data, MeigaralistPl
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import pandas as pd
import polars as pl

In [3]:
fp1 = DATA_DIR / "kessan.parquet"
fp2 = DATA_DIR / "meigaralist.parquet"
df1 = read_data(fp1)
KPL = KessanPl(df1)
df2 = read_data(fp2)
MPL = MeigaralistPl(df2)

In [4]:
# evaluation_dateで指定した日における、決算進捗率が取得可能な全銘柄の四半期決算進捗率をpl.DataFrameで作成し、返す
# 進捗率は、evaluation_date時における当期最新決算予想に対する四半期決算の進捗率。
# get_expected_quatery_settlements_progress_rate(self, valuation_date: date=date.today()) -> pl.DataFrame:
df = KPL.get_expected_quatery_settlements_progress_rate(ev_date)
df = df.filter(pl.col("code")==code)
df = df.select([
    "code",
    'yearly_settlement_date',
    "quater",
    "sales_pr(%)",
    "operating_income_pr(%)",
    "ordinary_profit_pr(%)",
    "final_profit_pr(%)"
])

pandas_df = df.to_pandas()

In [5]:
# def show_fig_performance_progress_rate_pycharts(pandas_df)
df = pandas_df
name = MPL.get_name(code)
rec_idx = df.shape[0] - 1
fyear = df.loc[rec_idx, "yearly_settlement_date"]
quater = df.loc[rec_idx, "quater"]

# グラフ出力オプション
pio.renderers.default = 'iframe'

# 出力グラフのplot設定(1行4列 -> 横並びに4つ表示)
specs = [
    [{"type": "pie"}, {"type": "pie"}, {"type": "pie"}, {"type": "pie"}]
]
fig = make_subplots(rows=1, cols=4, specs=specs)

# pychartオブジェクトのセット
for i in range(4):
    # pychartデータのセット(pandas.DataFrameにセットする)
    labels = ["進捗率(%)", " "]
    pr = df.loc[rec_idx, df.columns[i+3]]
    
    values = [pr, 100-pr]
    chart_df_data = {
        "labels": labels,
        "values": values
    }
    chart_df = pd.DataFrame(chart_df_data)

    # pychartオブジェクトの設定
    data_set = go.Pie(
        labels = chart_df["labels"],
        values = chart_df["values"],
        hole = 0.5,
        sort = False,
        marker = dict(colors=["aqua", "lightgrey"]),
        textinfo='percent',  # 全体の表示設定
        texttemplate=['%{percent}', '']
    )
    fig.add_trace(data_set, row=1, col=i+1)
    
# レイアウトの設定
items = ["売上高進捗率(%)", "営業利益進捗率(%)", "経常利益進捗率(%)", "純利益進捗率(%)"]
left_gap = 0.07
right_gap = 0.93
gap_correction = 0.01
gap = (right_gap - left_gap) / 3
annotations = []
for i in range(4):
    x = left_gap + gap * i
    if i == 1:
        x = x + gap_correction
    elif i == 2:
        x = x - gap_correction
    annotations.append(
        dict(text=items[i], x=x, y=0.5, font_size=12, showarrow=False)
    )

# 設定したレイアウトをpychartオブジェクトにセット
fig.update_layout(
    showlegend=False, # 凡例出力をoff
    annotations=annotations
)

# 出力
print(f'{name}({code})の{fyear.year}年{fyear.month}月期第{quater}四半期決算進捗率(評価日：{ev_date})')
fig.show()

ベルーナ(9997)の2025年3月期第2四半期決算進捗率(評価日：2024-11-16)


In [6]:
pandas_df

,code,yearly_settlement_date,quater,sales_pr(%),operating_income_pr(%),ordinary_profit_pr(%),final_profit_pr(%)
0,9997,2025-03-31,1,23.7,8.5,13.4,12.4
1,9997,2025-03-31,2,44.8,31.0,33.1,32.1


In [7]:
quater

np.int64(2)

In [8]:
print(type(fig))

<class 'plotly.graph_objs._figure.Figure'>
